In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install dwave-ocean-sdk

In [ ]:
!pip install dimod

In [ ]:
import pandas as pd
import dimod

# ==============================
# 1. LOAD DATASET DARI GDRIVE
# ==============================
file_path = "/content/drive/MyDrive/dataset/titanic_small.csv"
df = pd.read_csv(file_path)

print("Data Loaded:")
print(df.head())

# ==============================
# 2. PREPROCESSING
# ==============================
df = df.replace({True:1, False:0, 'TRUE':1, 'FALSE':0})

y = df["survived"]
X = df.drop("survived", axis=1)

# ==============================
# 3. HITUNG RELEVANCE
# ==============================
relevance = X.apply(lambda col: col.corr(y)).fillna(0)

print("\n=== RELEVANCE ===")
print(relevance)

# ==============================
# 4. HITUNG REDUNDANCY
# ==============================
redundancy = X.corr().fillna(0)

# ==============================
# 5. BANGUN QUBO
# ==============================
lambda_param = 0.5
features = list(X.columns)

Q = {}

for f in features:
    Q[(f, f)] = -relevance[f]

for i in range(len(features)):
    for j in range(i+1, len(features)):
        Q[(features[i], features[j])] = lambda_param * redundancy.iloc[i, j]

# ==============================
# 6. SOLVE DENGAN DIMOD
# ==============================
bqm = dimod.BinaryQuadraticModel.from_qubo(Q)

sampler = dimod.SimulatedAnnealingSampler()
sampleset = sampler.sample(bqm, num_reads=200)

best = sampleset.first.sample

# ==============================
# 7. OUTPUT HASIL
# ==============================
print("\n=== HASIL SELEKSI FITUR ===")
selected_features = [k for k,v in best.items() if v == 1]

print("Selected Features:", selected_features)
print("Energy:", sampleset.first.energy)

# ==============================
# 8. EVALUASI ML
# ==============================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

if len(selected_features) > 0:
    X_selected = X[selected_features]

    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.3, random_state=42)

    model = RandomForestClassifier()
    model.fit(X_train, y_train)

    acc = model.score(X_test, y_test)
    print("\nAccuracy:", acc)
else:
    print("\nTidak ada fitur terpilih")

Data Loaded:
   pclass  survived  sex  age  fare  cabin   miss  master     mr    mrs  \
0       1         1    1    2     2   True   True   False  False  False   
1       1         1    0    0     1   True  False    True  False  False   
2       1         0    1    0     1   True   True   False  False  False   
3       1         0    0    2     1   True  False   False   True  False   
4       1         0    1    2     1   True  False   False  False   True   

    rare  alone  port S  port C  port Q  
0  False   True    True   False   False  
1  False  False    True   False   False  
2  False  False    True   False   False  
3  False  False    True   False   False  
4  False  False    True   False   False  

=== RELEVANCE ===
pclass   -0.319979
sex       0.537719
age      -0.054794
fare      0.169683
cabin     0.314354
miss      0.299438
master    0.056279
mr       -0.536628
mrs       0.363751
rare     -0.005318
alone    -0.206754
port S   -0.175287
port C    0.219648
port Q   -0.067770

/tmp/ipykernel_10606/1705177264.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({True:1, False:0, 'TRUE':1, 'FALSE':0})



=== HASIL SELEKSI FITUR ===
Selected Features: ['age', 'alone', 'cabin', 'fare', 'master', 'miss', 'mr', 'mrs', 'pclass', 'port C', 'port Q', 'port S', 'rare', 'sex']
Energy: -2.8901789279709895

Accuracy: 0.7643312101910829
